# Projeto A2 — Designação com Método Húngaro

**Disciplina:** Pesquisa Operacional I  
**Trilha 2:** Transporte e designação  
**Cenário:** Viagem externa comercial (narrativa Mastercorp)

Documentação: `CENARIO-MASTERCORP.md`, `MODELAGEM-FORMAL.md`

In [ ]:
# Colab: descomente para instalar dependências
# !pip install -q matplotlib numpy seaborn

import os
os.environ.setdefault("MPLBACKEND", "Agg")

from pathlib import Path
BASE = Path('.')
if not (BASE / 'hungarian.py').exists():
    BASE = Path('/content')  # ajuste se fizer upload no Colab

import sys
sys.path.insert(0, str(BASE))

from hungarian import (
    carregar_matriz_csv,
    carregar_metadados,
    hungarian,
    forca_bruta,
    baseline_rotacao,
    custo_alocacao,
)
from run_projeto import plot_heatmap, plot_bipartido

DADOS = BASE / 'dados'
SAIDA = BASE / 'saida'
SAIDA.mkdir(exist_ok=True)

## 1. Cenário e dados

In [ ]:
meta = carregar_metadados(DADOS / 'consultores_blocos.json')
row_labels, col_labels, custo = carregar_matriz_csv(DADOS / 'matriz_custos_designacao.csv')

print(meta['meta']['empresa_narrativa'])
print('Consultores:', [c['id'] for c in meta['consultores']])
print('Blocos:', [b['id'] for b in meta['blocos_regionais']])
print('Matriz (h):')
for i, row in enumerate(custo):
    print(row_labels[i], row)

## 2. Método Húngaro (implementação manual)

Ver funções em `hungarian.py`: redução de linhas/colunas, matching em zeros, cobertura mínima, ajuste com $h$.

In [ ]:
resultado = hungarian(custo)
print('Custo ótimo:', resultado.custo_total, 'h')
print('Atribuição (índices):', resultado.atribuicao)
for i, j in resultado.atribuicao:
    print(f"  {row_labels[i]} -> {col_labels[j]} = {custo[i][j]:.2f} h")

print('\n--- Passos ---')
for p in resultado.passos:
    print('-', p.titulo, p.detalhe)

## 3. Validação (força bruta) e impacto vs baseline

In [ ]:
fb_pares, fb_custo = forca_bruta(custo)
assert abs(resultado.custo_total - fb_custo) < 1e-6

base = baseline_rotacao(len(custo), 1)
base_custo = custo_alocacao(custo, base)
economia = (base_custo - resultado.custo_total) / base_custo * 100

print('Força bruta:', fb_custo, 'h — OK')
print('Baseline rotação ingênua:', base_custo, 'h')
print(f'Economia: {economia:.1f}%')

## 4. Visualização (rubrica: heatmap + grafo bipartido)

In [ ]:
from IPython.display import Image, display

plot_heatmap(custo, row_labels, col_labels,
             'Matriz de custos', SAIDA / 'heatmap_original.png')
plot_heatmap(resultado.passos[2].matriz, row_labels, col_labels,
             'Matriz reduzida', SAIDA / 'heatmap_reduzida.png')
plot_heatmap(custo, row_labels, col_labels,
             'Solução ótima', SAIDA / 'heatmap_solucao.png',
             highlight=resultado.atribuicao)
plot_bipartido(row_labels, col_labels, resultado.atribuicao,
               SAIDA / 'grafo_bipartido.png')

for nome in ['heatmap_original.png', 'heatmap_reduzida.png',
             'heatmap_solucao.png', 'grafo_bipartido.png']:
    display(Image(filename=str(SAIDA / nome)))